# 11 - Pandas II: Exploring further tabular data


Original content from Felipe Alvarez de Toledo available at https://github.com/FelipeAdeT/PythonforHumanities

Content adaptation and modifications by Maroussia Bednarkiewicz.

### Navigation Reminder

- **Grey cells** are **code cells**. Click inside them and type to edit.
- **Run**  code cells by pressing $ \triangleright $  in the toolbar above, or press ``` shift + enter```.
-  **Stop** a running process by clicking &#9634; in the toolbar above.
- You can **add new cells** by clicking to the left of a cell and pressing ```A``` (for above), or ```B``` (for below).
- **Delete cells** by pressing ```X``` or the bin symbol in the right upper corner of the cell.
- Run all code cells that import objects (such as the one below) to ensure that you can follow exercises and examples.
- Feel free to edit and experiment - you will not corrupt the original files.

## Introduction

In the previous lesson, we learned how to access data contained within a DataFrame. Next, we learn to clean, restructure, combine, analyze and visualize tabular information with Pandas.

---

In [ ]:
# @title Grant GoogleColab access to your GoogleDrive and import questions for this notebook
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Add your module folder to Python path
import sys
module_path = f"/content/drive/My Drive/IDH/Notebooks"
sys.path.append(module_path)
print("GoogleColab can now access your GoogleDrive.")

# Import exercises
from QuestionsPandas2 import Q1, E1, E2, E3, E4, E5, question, solution

---
## Lesson goals

- Sort, add and remove columns and rows
- Perform math on columns and rows
- Understand Tidy Data and pivot using **df.melt()**
- Join and append tables through **df.merge()** and **df.append()**
- Understand basics of **visualizing DataFrames** in Pandas

**Key Concepts:** Tidy data, join, merge, melt

---


### Exercise 1

Using what you learned in previous lessons, go ahead and import Pandas as pd in the cell below.

In [ ]:
# import Pandas


Now, create a DataFrame called 'pg_authors' with the CSV file 'pg_authors.csv' in the folder called 'PG'. The file path is 'PG/pg_authors.csv'.

In [ ]:
# Import metadata from PG's authors


You should be able to run the following cells without getting an error message.

In [ ]:
# Set the column 'author_id' as index.
pg_authors.set_index('author_id', inplace=True)

In [ ]:
# Print the names of the columns
# 'author_id' is not a column anymore since it has been set as index
pg_authors.columns.tolist()

In [ ]:
# Transform the values for birth and death dates into integers
pg_authors["birth"] = pg_authors["birth"].astype("Int64")
pg_authors["death"] = pg_authors["death"].astype("Int64")

In [ ]:
solution(E1)

In the code cells below, examine the object in one of the ways discussed previously in the course. You can call its name, use the df.head() or df.tail() method, or the df.columns attribute. You can check the length of the dataframe with the len() function.

# Editing & cleaning DataFrames

Oftentimes, we will need to modify the columns and rows in a table in different ways. In this section, we will look at sorting dataframes, adding and removing columns, joining datasets and removing duplicate values.  

## Sorting DataFrames

```df.sort_values()``` is the method for sorting dataframes according to a column's contents. When applying this, we must note that it is not affecting the original dataframe. Thus, we either have to assign our modified dataframe to a variable or specify that we want the method to modify the original values "in place."

We should be familiar with assigning variables by now:

```python
df2 = df2.sort_values('column')
```

In this case, we are reassigning the original dataframe variable to its new value, a common approach.

Alternatively, through the in_place keyword, we can modify the dataframe directly.

```python
df2.sort_values('column', in_place=True)
```

The default is to sort by ascending order; to change to descending, we must specify ascending=False.

In our PGauthors DataFrame, we can sort authors by their death date.

In [ ]:
pg_authors.sort_values('death')

Note that this the line of code above will display the death date but it will not modify the dataframe. To do so we can assign the new sorted dataframe to a variable:

```
pd_authors_sorted = pg_authors.sort_values('death')
```

or use the parameter `in_place = True`:

```
pg_authors.sort_values('death', in_place = True)
```

## Adding and removing columns and rows

To add a column, just call an unassigned column name with bracket notation and specify the desired value for all cells.

```python
df['newname'] = x
```

To remove a column, use the df.drop() method:

```python
df = df.drop(columns = 'column_name')
```
Once again, you have to reassign the variable or specify in_place=True. You can include multiple columns by providing a list.

The same method applies for dropping rows, where instead of the keyword argument 'columns', you use the keyword 'index' and specify the indices of the rows you would like to remove.

```python
df = df.drop(index = [index numbers])
```
---

### Exercise 2

As a test, using the cells below, create a new column titled 'testcolumn' with a blank value ('') and then remove it. You can check the changes you make by calling the dataframe or the df.columns attribute.

In [ ]:
solution(E2)

# A Note on tidy data

**Tidy data** is the preferred format for many types of data analysis and visualization, including Pandas and multiple Python modules.

Datasets contain **values**, **variables** and **observations**.

>A dataset is a collection of **values**, usually either numbers (if quantitative) or strings (if qualitative). Values are organised in two ways. Every value belongs to a variable and an observation. A **variable** contains all values that measure the same underlying attribute (like height, temperature, duration) across units. An **observation** contains all values measured on the same unit (like a person, or a day) across attributes.

**In tidy data,**
- each **variable** forms one **column**,
- each **observation** forms a **row**, and
- each **observational unit** forms a **table**.

For example, in the file that contains PG's metadata, we have stored each text in a row (observations) and the information about each text in columns. The whole is our PG's texts metadata. And we created a separate file for authors with this time each author in a row (observations) and the information about the author in columns. The whole dataframe is our PG's authors' dataframe.

Often, datasets don't fulfill these requisites, with common problems being that:

- Column headers are values, not variable names (i.e., one variable is stored in several columns).
- Multiple variables are stored in one column.
- Variables are stored in both rows and columns.
- Multiple types of observational units are stored in the same table.
- A single observational unit is stored in multiple tables.

Analytically, a dataset like this:

| person | treatment | result |
|--|--|--|
|John|a|NaN|
|Jane|a|16|
|Mary|a|3|
|John|b|2|
|Jane|b|11|
|Mary|b|1|

Is better than one like this:

| person | treatment a| treatment b |
|--|--|--|
|John|NaN|2|
|Jane|16|11|
|Mary|3|1|

That is because in the second table the rows contain different types of observations, namely whether the person has received the treatment and the results obtained. Hence the columns with treatments a and b contain in fact two variables for treatments and results. In the first table, the two variables get two distinct columns. This facilitates, in turn, operations on the data in each column.

> Source:http://vita.had.co.nz/papers/tidy-data.pdf

We will often find untidy data structures in data that is intended for display, such as summary tables.

We won't get further into tidy data here, but a great resource are Eric Monson's talks, and his Github repository on Jupyter and Pandas, which includes two notebooks on Tidy Data. See https://github.com/emonson/pandas-jupyterlab/blob/master/TidyDataIntro.ipynb and https://github.com/emonson/pandas-jupyterlab/blob/master/TidyDataAdvanced.ipynb.

A very useful method for achieving a clean(er) dataset is the **``` df.melt()```** method. This method will unpivot columns into rows, thus helping reorganize values that have been erroneously included as variables.

![Pivot Table](https://pandas.pydata.org/pandas-docs/stable/_images/reshaping_melt.png)

Source: https://pandas.pydata.org/pandas-docs/stable/user_guide/reshaping.html

The documentation for this method gives you more details about the different parameters: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.melt.html.

The way scientists register data is often not appropriate to perform data analysis. When data analysts receive such untidy datasets, they constantly have to pivot tables. You can read more on Pandas' different methods to do so: https://pandas.pydata.org/pandas-docs/stable/user_guide/reshaping.html

```python
df.melt(id_vars = [list of variables to remain unchanged], value_vars = [list of variables to be made into values], var_name = 'nameforvariable', value_name = 'nameforvalues')
```

To clarify, in our example, the melt would look something like this:

```python
treatment_table = treatment_table.melt( id_vars = 'person', value_vars = ['treatment a', 'treatment b'], var_name = 'treatment', value_name = 'result')
```

In [ ]:
treatment_table = pd.DataFrame({'person':{0: 'John', 1: 'Jane',2: 'Mary'}, 'treatment a': {0: None, 1: 16, 2: 3}, 'treatment b': {0:2, 1:11,2:1}})

In [ ]:
treatment_table

In [ ]:
treatment_table.melt(id_vars = 'person', # Column used as identifier
                     value_vars = ['treatment a', 'treatment b'], # Variables to be melted into values
                     var_name = 'treatment', # Name for the column with the melted values
                     value_name = 'result') # Name of the column with the values from the variables in `value_vars`

### Exercise 3

We have a dataset that we need to format for analysis with Python or Tableau. It gives historical information about epidemic diseases in several ghettoes in the Krakow district in Poland during World War II.

The dataset is untidy. The observations about which disease was observed where are given in one column. Instead, we should have one column for the type of epidemic disease called **Epidemic** and one column to report whether the disease was observed or not, **EpidemicYN**.

Use `.melt()` to pivot this horizontal table and get a vertical one. The column with the **variables** that will be turned into **values**, `value_vars`, namely the type of epidemic diseases: 'UnspecE', 'UnknownE', 'UncertE', 'TyphusE', 'TyphoidE', 'TuberE', 'no_E', 'DysenteryE', will be called **Epidemic**. The name of the column with the actual values, namely the observations about the presence (Y) or absence (N) of the epidemic disease in the given ghetto, `value_name`, should be **EpidemicYN**.

In [ ]:
ghettoes = pd.read_csv(f"{module_path}/Other_files/KrakowGhettoes.csv")
ghettoes.head()

In [ ]:
solution(E3)

**Note on the index:**

In the previous notebook, we stressed the fact that the index column must be well chosen, especially when the table is then saved as CSV. We can observe that the dataset about the disease in the Krakow ghettoes did not get a proper index. The table was saved with a default index without a name. When we import the table with Pandas, the index becomes a column and receives automatically the title **Unnamed:0**. To prevent this we can reset the index or set the column 0 as the index when we import the file, like in the following cell with the parameter `index_col=0`:

In [ ]:
ghettoes = pd.read_csv(f"{module_path}/Other_files/KrakowGhettoes.csv", index_col=0)

## Joining or merging datasets

Often we will find ourselves with two related tables that we want to combine in order to analyze them together. We might have two tables with **different variables for the same observations**. Joining (as it is known in SQL [Structured Query Language]) or **merging** (as it is referred to in Pandas) is the action of combining two datasets that give different variables for the same observations. Joining two tables results in a table that adds variables (columns) to existing records.

Pandas documentation on merging methods: https://pandas.pydata.org/pandas-docs/stable/user_guide/merging.html

> Join: A join combines rows in one or more tables based on common values

We can **join** or **merge** two dataframes based on a common unique identifier (also known as a **key**): Python will identify rows in both tables with the same key and output a new table with variables from both tables for that key.

For two tables, left and right, there are 4 types of joins. These vary according to the priority given to the table and the treatment of non-matching values. Be conscientious about the type of join you employ, as the choice will determine the table(s) from which non-matching values will be dropped.

Suppose we have two tables, left and right:

|Types of Joins ||
--------------|-----|
|Left| Keeps all rows in the left table, and will join these to rows in the right table with a matching key.|
|Inner| Only gives us rows with matches in both tables (will also drop Left-only rows)|
|Outer| Returns all rows unique in each table and drops the common rows.|
|Right| Returns all rows in the right table, joined to any rows with a matching key from the left table.|



One method for joining dataframes is ```df.merge()```, and it is specified in this way:

```python
df3 = pd.merge(df1, df2, on='key_column_name', how='left')
```
The 'how' keyword specifies the type of join to execute, where df1 acts as the left table, and df2 acts as the right.

The keyword 'on' determines the name of the common key. If the names of the key columns are not the same in both tables, you can use left_on and right_on to specify.

Let's import two dataframes on which we can practice. We have stored all the subjects and bookshelves topics with the number of matching books from the Project Gutenberg metadata in two separate CSV files, **pg_subjects_counts.csv** and **pg_bookshelves_counts.csv**. They are stored in the folder **PG**.

In [ ]:
# Import the files into Pandas dataframes
pg_subjects = pd.read_csv(f"{module_path}/PG/pg_subjects_counts.csv")
pg_bookshelves = pd.read_csv(f"{module_path}/PG/pg_bookshelves_counts.csv")

In [ ]:
pg_subjects.head()

In [ ]:
pg_bookshelves.head()

Both dataframes have a column **Topics** and a column with the count of entries in the Project Gutenberg that match each topic.

We can display the length of each dataframe to know how many topics they have each.

In [ ]:
print(f"There are {len(pg_subjects)+1} unique topics in subjects")
print(f"There are {len(pg_bookshelves)+1} unique topics in bookshelves")

We can now merge the two dataframes into a new dataframe that we will call **pg_topics**. We will use the `outer` parameter to get only the topics that are *not* common to both subjects and bookshelves.

In [ ]:
pg_topics = pd.merge(
    pg_subjects,
    pg_bookshelves,
    on='Topics',
    how='outer'
).fillna(0)              # replaces empty values NaN with 0 for missing topics

In [ ]:
# Ensure counts are integers
pg_topics['Subjects_Count']    = pg_topics['Subjects_Count'].astype(int)
pg_topics['Bookshelves_Count']  = pg_topics['Bookshelves_Count'].astype(int)

In [ ]:
pg_topics

### Exercise 4 & Question 1

Since **pg_topics** only contains the so-called `outer` topics, namely those which are only in **Subjects** or in **Bookshelves**, we can calculate how many topics were dropped in our merging operation. Use the length of **pg_subjects**, **pg_bookshelves**, and **pg_topics** to find out how many common topics to Subjects and Bookshleves were dropped in our operation.

In [ ]:
question(Q1)

In [ ]:
solution(E4)

### Exercise 5

Let's find out the topics that are present in **Subjects** and **Bookshelves** using the `inner` parameter. Modify the code we used to merge **pg_subjects** with **pg_bookshelves** to create a new dataframe called **pg_topics_common** that will gather all the topics present in Subjects **and** Bookshelves.

In [ ]:
# Merge pg_subjects and pg_bookshelves to get the topics present in both
pg_topics_common =

In [ ]:
solution(E5)

In [ ]:
# Ensure counts are integers
pg_topics_common['Subjects_Count'] = pg_topics_common['Subjects_Count'].astype(int)
pg_topics_common['Bookshelves_Count'] = pg_topics_common['Bookshelves_Count'].astype(int)

In [ ]:
pg_topics_common

You can see that **pg_topics_common** has 66 rows, which means that there are 67 (66+1) topics present in Subjects and Bookshelves.

## Appending datasets

Alternatively, we might have two tables with the **same variables, and different observations**. In this case, we might want to add rows at the end of one table. For this, we **append**  or **concatenate** information to a DataFrame.

> **Appending** adds rows with like variables to a table.

To append two datasets together, that is, to add one table as new rows in another, you need two tables with the same column structure.

Then, you use the ```df.append()``` method:

```python
df1.append(df2)
```

This method also accepts dictionaries and other series, in lieu of a second dataframe.

The 'ignore_index' argument can also be useful for ignoring the indices from the individual DataFrames. For more information, consult the [documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.append.html).

If you have more than two DataFrames, a more general option is the ```pd.concat()```method.

```python
dfslist =[DataFrame1, DataFrame2, Dataframe3]

newdf = pd.concat(dfslist)
```

## Finding unique values

Another useful method, available for dataframes or series, is the option of finding only unique values.

The `series.unique()` method will return a list of unique values within a single column:

```python
df['column_name'].unique()
```

Let's explore further the Project Gutenberg metadata with this method:

In [ ]:
# Import PG's metadata in a Pandas dataframe
pg_metadata = pd.read_csv(f"{module_path}/PG/pg_catalog.csv", index_col="Text#")

With the `unique()` method we can obtain the unique languages present in the Project Gutenberg.

We can first select the languages.

In [ ]:
pg_languages = pg_metadata['Language']

In [ ]:
len(pg_languages)

**pg_languages** contains all the rows of **pg_metadata**, i.e. 76236 (76235 + 1) but there are not 76236 languages. We can use `.unique()` to get a list of each language.  

In [ ]:
# Create a Pandas object series with the unique languages
pg_languages = pg_languages.unique()
# Convert the object series into a list
pg_languages = pg_languages.tolist()

In [ ]:
len(pg_languages)

In [ ]:
pg_languages

In the list, we see languages appearing in pairs. These correspond to text in two languages present in the Project Gutenberg. Here is where Python and Pandas become extremely handy. In just one line (cell), we can create a list of all the unique languages from our metadata.

In [ ]:
pg_languages = (
    pg_metadata['Language']
      .dropna()               # drop NaNs (missing values)
      .str.split(';')         # split each cell into a list
      .explode()              # one item per row
      .str.strip()            # trim whitespace
      .unique()               # grab only distinct items
      .tolist()
)

In [ ]:
len(pg_languages)

We have thus 71 unique languages in the Project Gutenberg because Python counts the zero.

In [ ]:
pg_languages

Another powerful method, **```df.value_counts()```** allows us to get the number of text for each language and language pairs.

In [ ]:
pg_lang_counts = pg_metadata["Language"].value_counts().reset_index()

In [ ]:
pg_lang_counts

Again, Python and Pandas allow us to be more sophisticated and in just one line of code (cell) we can extract from PG's metadata all the languages and create a new table which count the number of texts for each language and gives us the unique IDs of each text. Run the next cells to see the results.

In [ ]:
pg_lang_counts = (
    pg_metadata
      .reset_index()   # bring Text# into a column
      .groupby('Language')
      .agg(
         language_count = ('Language', 'size'),
         text_ids=('Text#', lambda ids: ",".join(ids.astype(str)))
      )
      .reset_index()
)

In [ ]:
pg_lang_counts

You do not need to fully understand this level of code. We are presenting to you here advanced examples to give you an idea of what is feasible.

For example, if we are interested in a given language, we can now select the ids of all the texts for this language and get them from our metadata. We can look at the row for Spanish with the following line:

In [ ]:
pg_lang_counts[pg_lang_counts['Language']=='es']

# Math on DataFrames

Series and DataFrames have mathematical methods associated to them like ```sum()```,```mean()```, ```median()```,```max()```, ```min()```...

In Pandas, math is applied efficiently to columns or rows. It is applied down columns by default. To act upon rows, you must specify otherwise by including axis=1 inside the parentheses.

Strings are ignored or handled in a logical way. NaN/Null values are ignored by default, instead of causing errors.

Dataframes have mathematical methods:

```python
df.sum()
```

Let's take an example: We want to calculate the age of PG's authors. We will first upload our CSV files with the data from PG's authors.

In [ ]:
# Import the CSV with PG's authors
# Convert dates to integers
pg_authors = pd.read_csv(f"{module_path}/PG/pg_authors.csv",
                         dtype={'birth': 'Int64',
                                'death': 'Int64'}
)

In [ ]:
pg_authors.head()

Some authors do not have birth and death dates. We need to drop these rows to make our calculations.

In [ ]:
# Create a new dataframe called `pg_authors_ages`
# Drop rows without birth and/or death dates from pg_authors
pg_authors_ages = pg_authors.dropna(subset=['birth','death']).copy()

In [ ]:
print(f"{len(pg_authors)} - {len(pg_authors_ages)} = {len(pg_authors) - len(pg_authors_ages)}")

Only a quarter of PG's authors, 11290 out of 39079, have both birth and death dates.

Now we can calculate the age of the authors with birth and death dates. For that, we simply create a new column:

```python
df['newcolumn']
```

And we use operators to act upon values in the columns. For instance:
    
```python
df['newcolumn'] = df['column1'] - df['column2']
```
Which returns a series of answers (adds cells in the same row).

In our case, we assign to this new column using the equals sign `=`, the age that we calculate by subtracting the birth date to the death date for each author. Pandas will run the operation automatically, row by row.

In [ ]:
pg_authors_ages['age'] = pg_authors_ages['death'] - pg_authors_ages['birth']

In [ ]:
pg_authors_ages.head()

Now we can calculate the mean average age of PG's authors

In [ ]:
# Calculate the mean average age
print(pg_authors_ages['age'].mean())

We can also calculate the age of the youngest and oldest authors using the methods `min()` and `max()`:

In [ ]:
print(f"The author who died the youngest was {pg_authors_ages['age'].min()}.")
print(f"The author who died the oldest was {pg_authors_ages['age'].max()}.")

Obviously, no author could have published her/his work and died at 1 year old. This must be a mistake. We can check all the authors with an age below 15 for example:

In [ ]:
pg_authors_ages[pg_authors_ages['age'] < 15]

We see that there are authors who are marked as **active**. It might be that the dates correspond therefore to something else than their actual birth and death dates. If we were to work with this dataset we should check each author, and either correct the dates or remove them if we are not sure about their identity. The same applies to the age 120 and we could also check if all the authors' ages above 110 are accurate or mistakes.

# Basic plotting with Pandas

# Plotting functions

Dataframes also come with the plot() method, which allows us to create different types of graphs.

There are two ways to access the plot method, either including the graph type in parenthesis or including it with a dot:

```python
df.plot(kind='line', x='column', y=['column1', 'column2'])
df.plot.line(x='column',y=['column1', 'column2'])
```

Plotting in Pandas is like plotting in excel: one column will be X values, all others will be Y. There are different libraries for data visualisation and we will explore a couple of them.

If we specify X in pandas, it will by default take all other columns for Y.

Common Types of plots:
- Line Chart
- Bar Chart
- Histogram
- Box Plot

Let's create a simple visualisation for our data directly from our dataframe.

In [ ]:
# Chronological plot: Age vs Birth Year
# We create a new dataframe to sort the birth dates chronologically
df_chrono = pg_authors_ages.sort_values('birth')

In [ ]:
df_chrono.plot(kind='line', x='birth', y=['age'])

This graph gives us a quick glimpse into our data and can help us make decisions about the next steps to take in our research. We will not delve further into visualisation for now, but we have below a couple of cells to illustrate further how advanced visualisation techniques can be used in Python to plot data and display results in readable graphs.

For advanced visualisation, we need more libraries, that we need to import.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

We can now generate more sophisitcated plots, with dots instead of lines because the death dates of the authors should not be connected for instance.

In [ ]:
# Generate the plot to show authors' ages
plt.figure(figsize=(20,8))
plt.scatter(df_chrono['birth'], df_chrono['age'], s=25, alpha=0.6, label='Authors')
plt.grid(True, linestyle='--', alpha=0.5)
plt.xlabel('Birth Year')
plt.ylabel('Age at Death')
plt.title('Author Age')
plt.show()

We can further highlight trends and specific periods.

In [ ]:
df = df_chrono

plt.figure(figsize=(15,6))
plt.scatter(df['birth'], df['age'], s=25, alpha=0.6, label='Authors')
plt.grid(True, linestyle='--', alpha=0.5)

# Trend line
z = np.polyfit(df['birth'], df['age'], 1)
p = np.poly1d(z)
plt.plot(df['birth'], p(df['birth']), color='C1', linewidth=2,
         label=f"Trend: slope={z[0]:.2f}")

# Century shading
plt.axvspan(1800, 1899, color='C2', alpha=0.1)
plt.axvspan(1900, 1999, color='C3', alpha=0.1)

plt.xticks(rotation=45)
plt.xlabel("Birth Year")
plt.ylabel("Age at Death")
plt.title("Author Lifespans Over Time")
plt.legend()
plt.tight_layout()
plt.show()


The scatter‑plot, produced in just a few lines of `Pandas` and `Matplotlib`, illustrates how quickly we can turn raw metadata into a visual diagnostic: a glance is often enough to decide whether a variable is worth deeper analysis. Here, plotting authors’ ages against birth years reveals a modest upward trend, but, more importantly, it exposes the patchiness of the underlying dates and the heavy concentration of records in the 19th century.

Because a catalogue study is chiefly concerned with what texts are present (languages, genres, formats, topics), the precise age at death of each author adds little explanatory power: it neither predicts digitisation likelihood nor affects subject coverage. Its main value lies in alerting us to bias, e.g., the absence of living or recently deceased writers, which in turn shapes how representative the corpus can be for any longitudinal cultural analysis. Thus `Pandas` helps us decide early that, while age is an interesting sidebar, it is probably not a central variable for understanding Project Gutenberg’s collection.

We will do more such data exploration, but this gives a first idea of the possibilities.

For more, see [E. Monson's Pandas Workshops](https://github.com/emonson/pandas-jupyterlab)

---
Lesson Summary

- Data can be manipulated on Pandas in many ways, including on  sorting, adding and removing columns and rows and performing mathematical operations
-  Data should be in Tidy Format
- Columns can be transposed into rows (pivoted) using **df.melt()**
- Multiple tables can be joined (add columns) and appended (add rows) using **df.merge()** and **df.append()**
- DataFrames can be visualized using **df.plot**